# Файл для исторического эксперимента Qwen

Собираем единый JSON для Kaggle из фото, эталона и предсказаний Grounding DINO.

In [1]:
import sys
from pathlib import Path
from IPython.display import display

cwd = Path.cwd().resolve()
candidates = (cwd, *cwd.parents, Path("/kaggle/working/data-labeling-shad"))
ROOT = next((p for p in candidates if (p / "model_comparison/workflow.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Открой ноутбук из клона репозитория или склонируй его в /kaggle/working/data-labeling-shad")
sys.path.insert(0, str(ROOT))
print("Корень репозитория найден")

Корень репозитория найден


In [2]:
from model_comparison.workflow import DATA, load_validation
from model_comparison.workflow import baseline
import json

photos, golden = load_validation()
dino, _ = baseline("grounding_dino", photos)
print(f"Фото и предсказания сверены: {len(photos)} ID")

Фото и предсказания сверены: 400 ID


In [3]:
combined = [
    {"photo_id": pid, "url": photo["image"], "group": photo["group"],
     "boxes": golden[pid]["boxes"],
     "grounding_dino": {"boxes": dino[pid]["boxes"], "error": dino[pid]["error"]}}
    for pid, photo in photos.items()
]
published = json.loads((DATA / "butterfly_validation.json").read_text(encoding="utf-8"))
assert combined == published
out = ROOT / "runs/butterfly_validation.json"
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(combined, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print(f"{out.name}: {len(combined)} фото; содержимое совпадает с сохранённым файлом")

butterfly_validation.json: 400 фото; содержимое совпадает с сохранённым файлом
